# Edmonton Office Brochure Extraction Pipeline
## Google Colab Entry Point

In [ ]:
# Cell 1 — Install dependencies
!pip install pymupdf pytesseract Pillow rapidfuzz pydantic openpyxl requests python-dotenv pandas
!apt-get install -y tesseract-ocr -q

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clone repository and configure environment variables
import os, sys, getpass

REPO_DIR = '/content/brochure-pipeline'

# Always start from /content to prevent nested clone directories
os.chdir('/content')

# Remove any previous clone and start fresh
!rm -rf {REPO_DIR}
!git clone --branch claude/edmonton-brochure-extraction-2r7Bu \
    https://github.com/dionathan-santos/brochure.git {REPO_DIR}

# Switch to the cloned directory (absolute path)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Gemini API Key ────────────────────────────────────────────────────────
# Tries Colab Secrets first; falls back to a manual password prompt
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_MAPS_API_KEY')
    print("✅ GEMINI_API_KEY loaded from Secrets")
except Exception as e:
    print(f"⚠️  Secrets unavailable ({type(e).__name__}) — enter the key manually:")
    os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

# ── Anthropic API Key (optional fallback) ─────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    os.environ['ANTHROPIC_API_KEY'] = ''

# ── Google Drive paths ─────────────────────────────────────────────────────
os.environ['MASTER_INVENTORY_PATH'] = '/content/drive/MyDrive/Brochures/Master_Inventory.xlsx'
os.environ['AVAILABILITIES_PATH']   = '/content/drive/MyDrive/Brochures/Availabilities.xlsx'
os.environ['OUTPUT_DIR']            = '/content/drive/MyDrive/Brochures/Output'

# Final check
key = os.environ.get('GEMINI_API_KEY', '')
print(f'Directory : {os.getcwd()}')
print(f'GEMINI_API_KEY: {key[:8]}...' if key else '❌ Key is empty — check above')
print('✅ Ready to run!' if key else '❌ Fix the API key before continuing')

In [ ]:
# Cell 3b — Diagnostics: verify Gemini API key and list available models
# Run this cell if Cell 4 returns a 404 error to confirm which models are accessible
import requests, os

key = os.environ.get('GEMINI_API_KEY', '')
if not key:
    print("❌ GEMINI_API_KEY is not set — run Cell 3 first")
else:
    r = requests.get(
        f"https://generativelanguage.googleapis.com/v1beta/models?key={key}",
        timeout=15,
    )
    print(f"Status: {r.status_code}")
    if r.ok:
        models = [m['name'] for m in r.json().get('models', [])]
        flash = [m for m in models if 'flash' in m.lower()]
        print("✅ API accessible. Available flash models:")
        for m in flash:
            print(f"   {m}")
    else:
        print(f"❌ Error: {r.text[:500]}")
        print()
        print("If you see 'SERVICE_DISABLED' or 'API not enabled':")
        print("  → Go to: https://console.cloud.google.com/apis/library/generativelanguage.googleapis.com")
        print("  → Enable the 'Generative Language API' for the project linked to this key")
        print("If you see 'API_KEY_INVALID':")
        print("  → Generate a new key at: https://aistudio.google.com/apikey")

In [ ]:
# Cell 4 — Run the pipeline
# Set brochure_dir to the folder on Google Drive where the PDFs are stored
# Set brokerage to the brokerage name (e.g. CBRE, Colliers, Cushman)
from main import run_pipeline

output_path = run_pipeline(
    brochure_dir='/content/drive/MyDrive/Brochures/CBRE/',
    brokerage='CBRE'
)
print(f'Output saved: {output_path}')